# Master Dataset Agri

In [17]:
import pandas as pd
import os

# 1. Directory Setup
base_dir = r"D:\MS_Data_Science_Thesis"
clean_folder = os.path.join(base_dir, r"Data_Cleaning\Semi_Clean_Datasets")

# Create the final destination folder
final_folder = os.path.join(base_dir, r"Data_Cleaning\Final_Master_Dataset")
os.makedirs(final_folder, exist_ok=True)

# Define the temporal window
START_DATE = '2016-01-01'
END_DATE = '2025-12-31'

def execute_master_merge():
    print(f"--- Executing Master Merge ({START_DATE} to {END_DATE}) ---")
    
    # 2. Load and Trim the Anchor (Cotton)
    cotton_path = os.path.join(clean_folder, "cotton_engineered_I.csv")
    master_df = pd.read_csv(cotton_path)
    master_df.columns = master_df.columns.str.strip()
    master_df['Date'] = pd.to_datetime(master_df['Date'])
    master_df = master_df[(master_df['Date'] >= START_DATE) & (master_df['Date'] <= END_DATE)]
    
    # 3. Load Secondary Datasets
    print("Loading secondary engineered datasets...")
    
    # WTI Crude (Fixing the lowercase 'date' issue on the fly)
    wti = pd.read_csv(os.path.join(clean_folder, "oil_engineered_III.csv"))
    wti.columns = wti.columns.str.strip()
    if 'date' in wti.columns:
        wti = wti.rename(columns={'date': 'Date'})
    wti['Date'] = pd.to_datetime(wti['Date'])
    
    # DXY
    dxy = pd.read_csv(os.path.join(clean_folder, "dxy_engineered_IV.csv"))
    dxy.columns = dxy.columns.str.strip()
    dxy['Date'] = pd.to_datetime(dxy['Date'])
    
    # Sentiment
    sentiment = pd.read_csv(os.path.join(clean_folder, "agri_sentiment_engineered_II.csv"))
    sentiment.columns = sentiment.columns.str.strip()
    sentiment['Date'] = pd.to_datetime(sentiment['Date'])
    
    # WASDE
    wasde = pd.read_csv(os.path.join(clean_folder, "wasde_engineered_V.csv"))
    wasde.columns = wasde.columns.str.strip()
    wasde['Date'] = pd.to_datetime(wasde['Date'])
    
    # Climate
    climate = pd.read_csv(os.path.join(clean_folder, "climate_engineered_VI.csv"))
    climate.columns = climate.columns.str.strip()
    climate['Date'] = pd.to_datetime(climate['Date'])

    # 4. Execute the Left Joins
    print("Executing Left Joins anchored to the Cotton trading calendar...")
    master_df = pd.merge(master_df, wti, on='Date', how='left')
    master_df = pd.merge(master_df, dxy, on='Date', how='left')
    master_df = pd.merge(master_df, sentiment, on='Date', how='left')
    master_df = pd.merge(master_df, wasde, on='Date', how='left')
    master_df = pd.merge(master_df, climate, on='Date', how='left')
    
    # Sort chronologically to be absolutely certain
    master_df = master_df.sort_values('Date').reset_index(drop=True)
    
    # 5. The Final Forward-Fill
    # This patches the 2 missing DXY days and any other minor calendar misalignments
    print("Applying final forward-fill to patch minor calendar gaps...")
    master_df = master_df.ffill()
    
    # 6. Drop Initial NaNs
    # The very first few rows of the 2016 dataset might still have NaNs from rolling windows 
    # that started on Jan 1st. We drop these to ensure a pristine tensor.
    initial_rows = len(master_df)
    master_df = master_df.dropna()
    final_rows = len(master_df)
    print(f"Dropped {initial_rows - final_rows} rows at the start of 2016 due to initial rolling window NaNs.")
    
    # 7. Generate the Target Variable (t+1)
    # Now that the grid is perfectly solid, we can safely shift the target
    print("Generating Target Variable (Target_Log_Return_t+1)...")
    master_df['Target_Log_Return_t+1'] = master_df['Log_Return'].shift(-1)
    
    # Drop the very last row, as we cannot predict t+1 for the present day yet
    master_df = master_df.dropna(subset=['Target_Log_Return_t+1'])
    
    # 8. Export the Ultimate Dataset
    output_path = os.path.join(final_folder, "Cotton_Master_Dataset.csv")
    master_df.to_csv(output_path, index=False)
    
    print("-" * 50)
    print(f"SUCCESS! The Master Dataset is complete.")
    print(f"Total Features: {len(master_df.columns)}")
    print(f"Total Trading Days Ready for Deep Learning: {len(master_df)}")
    print(f"Saved to: {output_path}")
    
    # Quick sanity check on missing values
    missing_data = master_df.isna().sum().sum()
    print(f"Total Missing Values in Final Tensor: {missing_data}")

if __name__ == "__main__":
    execute_master_merge()

--- Executing Master Merge (2016-01-01 to 2025-12-31) ---
Loading secondary engineered datasets...
Executing Left Joins anchored to the Cotton trading calendar...
Applying final forward-fill to patch minor calendar gaps...
Dropped 53 rows at the start of 2016 due to initial rolling window NaNs.
Generating Target Variable (Target_Log_Return_t+1)...
--------------------------------------------------
SUCCESS! The Master Dataset is complete.
Total Features: 37
Total Trading Days Ready for Deep Learning: 2963
Saved to: D:\MS_Data_Science_Thesis\Data_Cleaning\Final_Master_Dataset\Cotton_Master_Dataset.csv
Total Missing Values in Final Tensor: 0
